# Investigação da *baseline* de inferência e estudo de caso

## Introdução

Este *notebook* investiga a *baseline* de inferência implementada no *script* `src/03_cpu_gpu_baseline.py`, tomando como referência a revisão identificada na tabela de informações sobre a *baseline*.

• **Objetivo da investigação**

O objetivo é compreender o funcionamento do código, analisar as escolhas de implementação e os resultados de desempenho, além de reunir evidências que fundamentem decisões técnicas futuras. O documento também contempla um estudo de caso voltado à avaliação da capacidade de processamento e dos custos de infraestrutura para atender à demanda diária de um cliente.

• **Seções**

A investigação é estruturada em **oito seções**:

Nas **seções 1 a 4**, o foco é aplicado sobre o funcionamento da inferência e inclui a reconstrução do fluxo de execução, análise dos parâmetros de geração, investigação do aquecimento (*warmup*) e da metodologia de cronometragem, além do processo de obtenção, armazenamento e carregamento do modelo.

As **seções 5 e 6** tratam sobre memória e desempenho, apresentando a interpretação das métricas de consumo de RAM e VRAM, cálculo dos indicadores de desempenho (*throughput* e *speedup*) e comparação entre as três configurações de execução da inferência.

Por fim, as **seções 7 e 8** focam em reprodutibilidade e a aplicação prática, onde se examina o aprimoramento da persistência e a rastreabilidade dos resultados e um estudo de caso que avalia a viabilidade técnica e econômica do processamento diário de 10 mil clientes, a partir de um *benchmark* com 24 amostras.

• **Subseções**

Cada subseção apresenta a questão técnica que está sendo investigada, seguida pela apresentação dos conceitos, componentes do código envolvidos, as referências consultadas e as verificações realizadas. Quando aplicável, também são registradas as hipóteses formuladas antes dos experimentos, as evidências obtidas, a interpretação dos resultados e as decisões tomadas decorrentes da análise.

Também são documentadas as limitações das conclusões, as dúvidas ainda não esclarecidas e as eventuais alterações realizadas no código, distinguindo os **resultados originalmente publicados** daqueles produzidos durante esta investigação. Dessa forma, busca-se preservar a rastreabilidade do estudo e facilitar sua consulta, reprodução e revisão futura.

## Registros e controle de alterações

• ***Commit* mais recente do repositório no início da investigação**

[9a8ba49810d4b162f9eee03577720f0920ba8a06](https://github.com/leandrocsiqueira/ai-coding-tokenomics/commit/9a8ba49810d4b162f9eee03577720f0920ba8a06)

• **Estado do repositório local durante a revisão da seção 1**

A primeira versão deste *notebook* foi publicada no **GitHub** antes da inclusão deste registro de diagnóstico do **Git**. Consequentemente, a saída apresentada abaixo não representa necessariamente o estado do repositório no início da investigação.

O registro foi incorporado durante o refinamento da seção 1 para documentar a *branch*, o último *commit* e as alterações locais existentes no momento da execução da célula.

A revisão-base utilizada para estudar o funcionamento da inferência permanece identificada pelo *commit* 3a1be993041ed75f77f26f23b0cacf0aff507d30. As modificações realizadas posteriormente no *notebook* fazem parte da evolução do material de estudo.

In [4]:
import subprocess
from datetime import datetime
from pathlib import Path

raiz = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], check=False, capture_output=True, text=True
)

if raiz.returncode != 0:
    print("Nenhum repositório Git encontrado.")
else:
    projeto = Path(raiz.stdout.strip()).name

    branch = subprocess.run(
        ["git", "branch", "--show-current"], check=False, capture_output=True, text=True
    ).stdout.strip()

    commit = subprocess.run(
        ["git", "log", "-1", "--format=%h - %s"],
        check=False,
        capture_output=True,
        text=True,
    ).stdout.strip()

    status = subprocess.run(
        ["git", "-C", raiz.stdout.strip(), "status", "--short"],
        check=False,
        capture_output=True,
        text=True,
    ).stdout.strip()

    print(f"Projeto: {projeto}")
    print(f"Branch: {branch}")
    print(f"Último commit: {commit}")
    print(f"Estado: {'Alterações pendentes' if status else 'Limpo'}")

    if status:
        print(f"\nArquivos alterados:\n{status}")

print(
    "\nRegistro realizado em:",
    datetime.now().astimezone().isoformat(timespec="seconds"),
)


Projeto: ai-coding-tokenomics
Branch: main
Último commit: 9a8ba49 - fix: add PyTorch CUDA index to dependencies
Estado: Alterações pendentes

Arquivos alterados:
M notebooks/03_estudo_inferencia_01c.ipynb
?? notebooks/01c_Inferencia/
?? notebooks/versao_de_trabalho.ipynb

Registro realizado em: 2026-11-21T21:37:45-03:00


**• Informações sobre a *baseline***

| Item | Identificação |
|---|---|
| Repositório | [AI Coding Tokenomics](https://github.com/leandrocsiqueira/ai-coding-tokenomics) |
| *Script* investigado | [src/03_cpu_gpu_baseline.py](https://github.com/leandrocsiqueira/ai-coding-tokenomics/blob/3a1be993041ed75f77f26f23b0cacf0aff507d30/src/03_cpu_gpu_baseline.py) |
| CSV publicado | [results/cpu_gpu_baseline.csv](https://github.com/leandrocsiqueira/ai-coding-tokenomics/blob/3a1be993041ed75f77f26f23b0cacf0aff507d30/results/cpu_gpu_baseline.csv) |
| *Commit* da revisão-base | [3a1be993041ed75f77f26f23b0cacf0aff507d30](https://github.com/leandrocsiqueira/ai-coding-tokenomics/tree/3a1be993041ed75f77f26f23b0cacf0aff507d30) |
| Data de criação da revisão-base | 05 de outubro de 2026 |

• **Arquivos usados no estudo de caso**

| Item | Identificação |
|---|---|
| Caso do cliente | [01c_Inferencia/docs/CASO_CLIENTE_INFRAESTRUTURA_01c.md](./01c_Inferencia/docs/CASO_CLIENTE_INFRAESTRUTURA_01c.md) |
| Guia do *dataset* | [01c_Inferencia/docs/GUIA_DO_DATASET_01c.md](./01c_Inferencia/docs/GUIA_DO_DATASET_01c.md) |
| Localização do *dataset* | [01c_Inferencia/dataset/consumo_cartao_credito_01c.json](./01c_Inferencia/dataset/consumo_cartao_credito_01c.json) |
| *Script* de carregamento do *dataset* | [01c_Inferencia/dataset/loader_dataset.py](./01c_Inferencia/dataset/loader_dataset.py) |
| Referências AWS | [01c_Inferencia/dataset/referencias_aws_2026-10-19.json](./01c_Inferencia/dataset/referencias_aws_2026-10-19.json) |

• **Ambiente usado nessa investigação**

In [5]:
import platform
from datetime import datetime

import psutil
import torch
import transformers

print("=== EXECUÇÃO ===")
print(f"Data e hora: {datetime.now().astimezone():%d/%m/%Y %H:%M:%S %Z}")

print("\n=== HARDWARE E SISTEMA ===")
print(f"Sistema operacional: {platform.platform()}")

with open("/proc/cpuinfo", encoding="utf-8") as arquivo:
    for linha in arquivo:
        if linha.startswith("model name"):
            print(f"Processador: {linha.split(':', 1)[1].strip()}")
            break

print(f"RAM total: {psutil.virtual_memory().total / 1024**3:.2f} GiB")

if torch.cuda.is_available():
    print("\n=== GPU ===")
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(
        f"VRAM total: {torch.cuda.get_device_properties().total_memory / 1024**3:.2f} GiB"
    )
    print(f"CUDA do PyTorch: {torch.version.cuda or 'Não disponível'}")

print("\n=== SOFTWARE ===")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")


=== EXECUÇÃO ===
Data e hora: 21/11/2026 21:37:45 -03

=== HARDWARE E SISTEMA ===
Sistema operacional: Linux-7.0.0-43-generic-x86_64-with-glibc2.39
Processador: 12th Gen Intel(R) Core(TM) i7-12800H
RAM total: 31.03 GiB

=== GPU ===
GPU: NVIDIA RTX A3000 12GB Laptop GPU
VRAM total: 11.63 GiB
CUDA do PyTorch: 13.2

=== SOFTWARE ===
Python: 3.12.3
PyTorch: 2.15.0+cu132
Transformers: 5.22.2


• **Registros de alterações**

Nenhuma alteração realizada.

## Seção 1. Fluxo de execução e inferência

**Controles**

* ***Status*:** [x] Concluída sem dúvidas [ ] Concluída com dúvida [ ] Bloqueada

* **Alterações realizadas no código:** Nenhuma alteração é necessária

### 1.1 - Como se dá o fluxo de execução do *script*?

O diagrama a seguir apresenta o fluxo de execução do script, desde a inicialização do programa até o armazenamento dos resultados em arquivo CSV.

A representação identifica as principais funções, suas responsabilidades e as relações entre as etapas de **preparação**, **inferência**, **medição** e **persistência**. Também detalha o funcionamento da função *`run_inference()`*, utilizada nas três configurações previstas: **CPU FP32**, **CUDA FP32** e **CUDA FP16**.

<img src="./01c_Inferencia/images/01c_Inferencia_diagrama_geral.svg">

O fluxo representado evidencia a separação entre a inicialização do programa, a execução das inferências e o registro dos resultados. O *tokenizer* é carregado no escopo global, antes da chamada de *`main()`*, enquanto o modelo é carregado individualmente em cada execução de *`run_inference()`*, conforme o dispositivo e a precisão numérica selecionados. A função também concentra a preparação das entradas, o aquecimento, a geração medida, a coleta das métricas e a liberação dos recursos utilizados.

A execução começa obrigatoriamente pela configuração **CPU/FP32** e, quando **CUDA** está disponível, prossegue com **CUDA/FP32** e **CUDA/FP16**. As métricas retornadas por cada chamada são reunidas em *`main()`* e posteriormente gravadas em um arquivo **CSV**, estabelecendo um procedimento comum para comparar as configurações.

### 1.2 - Onde o *tokenizer* é criado no código?

O diagrama abaixo isola a inicialização do *tokenizer*, destacando que ela ocorre no **escopo global do módulo, antes da chamada a *`main()`***. Também mostra como a mesma instância é reutilizada durante as diferentes execuções de *`run_inference()`*.

<img src="./01c_Inferencia/images/01c_Inferencia_tokenizacao.svg">

O aspecto central é a diferenciação entre criação e uso do *tokenizer*. A chamada *`AutoTokenizer.from_pretrained(MODEL_IDENTIFIER)`* ocorre uma única vez durante a inicialização do módulo, independentemente de *`main()`* ser executada. Posteriormente, as chamadas de *`run_inference()`* reutilizam essa instância para tokenizar o prompt e decodificar os *tokens* gerados.

### 1.3 - Onde o modelo é usado no código e quais as responsabilidades dos demais componentes durante a execução da inferência?

O modelo é efetivamente usado para gerar respostas nas chamadas feitas à função *`generate()`*, presentes em duas funções:

• **`pre_inference_warmup()`:** chama o modelo para realizar uma geração de até 5 novos IDs de *token* com o propósito de aquecimento (*warmup*). A geração é descartada e essa execução não entra na cronometragem.

• **`execute_inference_with_timing()`:** chama o modelo para gerar até 50 novos tokens. Dessa execução, são obtidos a saída usada na avaliação e o tempo decorrido.

As demais partes do programa organizam essa utilização da seguinte forma:

| Etapa                               	| Parte do programa                                           	| Responsabilidade                                                                                                                                      	|
|-------------------------------------	|-------------------------------------------------------------	|-------------------------------------------------------------------------------------------------------------------------------------------------------	|
| Preparação do *tokenizer*             	| *`AutoTokenizer.from_pretrained()`*                             	| Carrega o *tokenizer* usado para converter o *prompt* em entradas numéricas e, posteriormente, converter os IDs da saída em texto.                        	|
| Preparação da memória GPU           	| *`prepare_cuda_memory_for_inference()`*                         	| Executa a coleta de lixo, limpa o *cache* **CUDA** não utilizado e redefine as estatísticas de pico antes do carregamento da configuração GPU.              	|
| Preparação do modelo e das entradas 	| *`run_inference()`*                                             	| Carrega o modelo no *`dtype`* solicitado, move-o para o dispositivo, ativa *`eval()`*, "tokeniza" o *prompt* e coloca os tensores no mesmo dispositivo do modelo. 	|
| Medição de tempo                    	| *`execute_inference_with_timing()`*                             	| Cronometra a geração medida. Em **CUDA**, sincroniza o dispositivo antes de iniciar o cronômetro e depois da geração, antes de encerrá-lo.                	|
| Cálculo das métricas                	| *`run_inference()`* e *`collect_gpu_memory_metrics()`*              	| Calculam *tokens* gerados e *tokens* por segundo; coletam **RSS** do processo e, na **GPU**, nome do dispositivo, pico de memória alocada e memória reservada.          	|
| Apresentação e registro             	| *`print_inference_results()`* e *`save_inference_results_to_csv()`* 	| Exibem métricas e texto no terminal e gravam as métricas no **CSV**. O texto gerado não é salvo no **CSV**.                                                   	|
| Liberação de recursos               	| Bloco *`finally`* de *`run_inference()`*                            	| Exclui referências ao modelo, às entradas e à saída, executa a coleta de lixo e, quando aplicável, limpa o *cache* **CUDA** não utilizado.                  	|

Assim, carregar o modelo não é o mesmo que executar a geração. O carregamento e a preparação acontecem antes; a geração ocorre em *`generate()`*; a interpretação e o registro dos resultados acontecem depois.

### 1.4 - O que entra e sai da função *`generate()` e como IDs se tornam textos*?

Na execução medida, a chamada à função *`generate()`* é feita da seguinte maneira:

```python

# A constante MAX_NEW_TOKENS é igual a 50

generated_ids = inference_model.generate(**input_data, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)

```

**O que entra pela função?** 

O conteúdo de *`PROMPT`* é primeiro convertido em tensores **PyTorch**:

```python

input_data = tokenizer(PROMPT, return_tensors="pt").to(execution_device)

```

O `**` na frente de **`input_data`** realiza o desempacotamento das entradas fornecidas pelo *tokenizer*, incluindo:

• **`input_ids`:** tensor de IDs numéricos dos *tokens* que representam o *prompt*.

• **`attention_mask`:** tensor que indica quais posições da entrada devem ser consideradas pelo modelo.

Esses tensores ficam no mesmo dispositivo do modelo. Os IDs são valores inteiros: a escolha de FP32 ou FP16 para o modelo não transforma os IDs em números de ponto flutuante.

Além das entradas "tokenizadas", o modelo também recebe:

• **A estratégia de geração:** *`do_sample=False`* desativa a amostragem na escolha dos próximos *tokens*.

• **Número máximo de novos *tokens*:** O parâmetro *`max_new_tokens=MAX_NEW_TOKENS`* estabelece como 50 o limite de novos tokens, sem incluir os tokens do *prompt*. Entretanto, a geração pode terminar antes desse limite.


**Nota**

A geração de aquecimento (*warmup*) recebe as mesmas entradas, mas define o parâmetro *`max_new_tokens`* com o valor da constante *`WARMUP_MAX_NEW_TOKENS`* (que é 5).


**O que sai da função?**

No *script*, a função *`generate()`* retorna um tensor de IDs de *tokens*, armazenado em *`generated_ids`*. Para o modelo usado, a sequência retornada contém os *tokens* da entrada seguidos dos novos *tokens*:

```text

generated_ids: [ IDs do prompt | IDs dos tokens gerados ]

```

Como há apenas um *prompt* por chamada, o tensor possui o seguinte formato:

```text

[1, quantidade de tokens de entrada + quantidade de tokens gerados]

```

Por isso, a quantidade de *tokens* novos é calculada desta maneira:

```python

generated_token_count = (generated_ids.shape[-1] - input_token_count)

```

**Como os IDs se tornam texto?**

A conversão para texto acontece separadamente:

```python

inference_result = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

```

Nessa operação:

• *`generated_ids[0]`* seleciona a única sequência do lote de processamento.
• *`tokenizer.decode()`* chama o tokenizer junto ao método *`decode()`* para transformar a sequência de IDs de *token* já produzida em texto legível.
• *`skip_special_tokens=True`* não inclui os *tokens* especiais na representação textual.

Como o *script* decodifica a sequência inteira, o texto exibido inclui a representação textual dos *tokens* do *prompt* e da continuação gerada, e não apenas a resposta nova. A contagem de *`output_tokens`*, por outro lado, considera somente os novos *tokens* acrescentados pela geração.

<img src="./01c_Inferencia/images/01c_Inferencia_entra_sai_modelo.svg">

## 2. Decisões sobre a Geração

### O efeito do parâmetro `do_sample`

No experimento anterior, `generate()` recebeu os IDs de 13 tokens de entrada e retornou uma sequência de 63 IDs, dos quais 50 correspondiam à continuação gerada. A decodificação permitiu ler essa continuação como texto. Resta examinar **como o modelo escolhe cada novo ID**. No [baseline](https://github.com/leandrocsiqueira/ai-coding-tokenomics/blob/3a1be993041ed75f77f26f23b0cacf0aff507d30/src/03_cpu_gpu_baseline.py), a chamada utiliza explicitamente `do_sample=False` e `max_new_tokens=50` tanto no aquecimento quanto na geração medida.

A cada passo da geração autorregressiva, o modelo produz *logits*, ou seja, pontuações não normalizadas associadas aos tokens candidatos do vocabulário. Antes da seleção, essas pontuações podem sofrer ajustes, como uma penalidade de repetição. A estratégia de geração determina como se obtém o próximo ID a partir das pontuações resultantes. Com `num_beams=1`, que é a configuração deste experimento, distinguem-se dois procedimentos:

🟣 **`do_sample=False` - decodificação gulosa (*greedy decoding*).** A cada passo, é selecionado o token com a maior pontuação ajustada (`argmax`). Não há sorteio. Nas mesmas condições de execução, tende a reproduzir a mesma sequência, mas diferenças numéricas ou de implementação entre ambientes podem afetar o resultado. A estratégia escolhe o token mais bem pontuado **em cada passo**, não necessariamente a sequência completa de maior probabilidade.

🔵 **`do_sample=True` - amostragem probabilística (*multinomial sampling*).** As pontuações ajustadas são transformadas em probabilidades, e o próximo token é sorteado segundo essa distribuição, possivelmente após filtros. Portanto, outros tokens além do mais provável podem ser escolhidos. Execuções com a mesma entrada podem produzir continuações diferentes. Fixar uma semente (*seed*) (*seed*) de aleatoriedade ajuda a reproduzir um sorteio nas mesmas condições, mas não transforma a estratégia em decodificação gulosa.

#### Configurações que também influenciam a escolha

O comportamento de `generate()` não depende apenas de `do_sample`. A [configuração de geração](https://huggingface.co/docs/transformers/v5.20.2/en/main_classes/text_generation#transformers.GenerationConfig) carregada com o modelo fornece valores que não forem substituídos na chamada. O arquivo [`generation_config.json` do Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/blob/main/generation_config.json) estabelece, na versão examinada, `do_sample=True`, `repetition_penalty=1.1`, `temperature=0.7`, `top_k=20` e `top_p=0.8`. A chamada explícita com `do_sample=False` no baseline **substitui o valor padrão de `do_sample`**, sem anular automaticamente os demais ajustes aplicáveis.

`repetition_penalty=1.1` altera as pontuações de tokens já presentes no contexto e pode influenciar **tanto a seleção gulosa quanto a amostragem**; o valor neutro é `1.0`. Já `temperature=0.7` modifica a concentração da distribuição de probabilidades, `top_k=20` restringe os candidatos aos 20 tokens com maior pontuação e `top_p=0.8` seleciona um subconjunto com massa acumulada de probabilidade de pelo menos 0,8, conforme a ordenação dos candidatos. **Esses três controles são usados na amostragem, mas não na decodificação gulosa.** A combinação de `top_k` e `top_p` pode restringir ainda mais o conjunto de candidatos; ela não determina antecipadamente qual token será sorteado.

O parâmetro `num_beams` também importa: `num_beams=1` permite comparar diretamente *greedy decoding* e amostragem multinomial; valores maiores ativam estratégias de busca em feixe e mudam a interpretação. Já `max_new_tokens=50` controla o **limite** de tokens novos, e não o método de escolha: a geração pode terminar antes, por exemplo, ao produzir um ID de fim de sequência (`eos_token_id`).

Na saída registrada na seção 1, o último ID foi `5766`. A configuração consultada lista `151645` e `151643` como IDs de fim de sequência; portanto, **aquele resultado específico não terminou com um EOS**, mas atingiu o limite de 50 novos tokens. A célula seguinte verifica novamente essa conclusão com os valores carregados no próprio ambiente.

Os dois exemplos abaixo reutilizam `inference_model`, `tokenizer`, `input_data`, `input_token_count` e `MAX_NEW_TOKENS` definidos na seção anterior. A configuração será primeiro inspecionada **em tempo de execução**, pois uma revisão diferente dos arquivos do modelo pode fornecer outros valores. Em seguida, compararemos o texto e os IDs resultantes de cada estratégia, sem utilizar a duração das execuções como benchmark.

In [6]:
generation_config = inference_model.generation_config

print("\n=== CONFIGURAÇÃO DE GERAÇÃO CARREGADA ===")

for parameter_name in (
    "do_sample",
    "num_beams",
    "repetition_penalty",
    "temperature",
    "top_k",
    "top_p",
    "eos_token_id",
):
    print(f"{parameter_name}: {getattr(generation_config, parameter_name)}")

eos_token_ids = generation_config.eos_token_id
if isinstance(eos_token_ids, int):
    eos_token_ids = [eos_token_ids]
elif eos_token_ids is None:
    eos_token_ids = []

last_new_token_id = int(new_token_ids[0, -1].item())
print(f"\nÚltimo ID novo da seção 1: {last_new_token_id}")
print(f"Esse ID é um EOS configurado? {last_new_token_id in eos_token_ids}\n")


NameError: name 'inference_model' is not defined

### Exemplo 1 - `do_sample=False`: escolha gulosa

Este exemplo reproduz a estratégia usada pelo baseline: com `num_beams=1` e `do_sample=False`, a escolha do token é feita pelo maior logit ajustado, sem amostragem. Mantemos o mesmo modelo, *prompt*, dispositivo e limite de 50 tokens. O programa compara os IDs retornados com `generated_ids` da seção 1 para verificar se a sequência foi reproduzida nesta execução.

A penalidade de repetição configurada no modelo pode continuar atuando; `temperature`, `top_k` e `top_p` não são empregados como filtros de amostragem nesta estratégia.

In [ ]:
with torch.inference_mode():
    greedy_generated_ids = inference_model.generate(
        **input_data,
        do_sample=False,
        num_beams=1,
        max_new_tokens=MAX_NEW_TOKENS,
    )

greedy_new_ids = greedy_generated_ids[:, input_token_count:]
greedy_text = tokenizer.decode(greedy_new_ids[0], skip_special_tokens=True)

print("\n=== EXEMPLO 1: do_sample=False ===")
print(f"Tokens novos: {greedy_new_ids.shape[-1]}")
print(f"IDs gerados: {greedy_new_ids[0].tolist()}")
print(f"Mesmo resultado da seção 1? {torch.equal(greedy_generated_ids, generated_ids)}")
print(f"\nTexto decodificado:\n{greedy_text}\n")



=== EXEMPLO 1: do_sample=False ===
Tokens novos: 50
IDs gerados: [9660, 2022, 374, 279, 1882, 315, 14719, 1495, 264, 8500, 315, 4244, 1119, 3842, 11211, 11, 1741, 438, 11931, 323, 61503, 15423, 13, 1096, 6147, 369, 8661, 8692, 323, 6358, 315, 1467, 821, 553, 17592, 11980, 323, 24588, 22414, 8153, 2878, 432, 13, 3216, 14719, 705, 1293, 9069, 315, 5766]
Mesmo resultado da seção 1? True

Texto decodificado:
 Tokenization is the process of breaking down a sequence of words into individual tokens, such as letters and punctuation marks. This allows for easier processing and analysis of text data by removing noise and identifying meaningful units within it. By breaking up long strings of characters



### Exemplo 2 - `do_sample=True`: amostragem probabilística

Agora o `do_sample` é modificado para `True`, mantendo `num_beams=1`, os mesmos dados de entrada e `max_new_tokens=50`. Os valores de `temperature`, `top_k`, `top_p` e `repetition_penalty` são herdados da configuração carregada do modelo, sem ajustes adicionais no código. Assim, **a mudança experimental explícita entre as chamadas de geração é `do_sample`**; os parâmetros da configuração passam a exercer efeitos distintos, de acordo com a estratégia.

Uma semente de aleatoriedade é fixada antes da primeira geração amostrada para permitir sua reprodução sob condições equivalentes. Uma segunda geração, com *seed*, serve para verificar se o sorteio produz uma continuação diferente. Resultados diferentes são possíveis, mas **não são garantidos** em toda combinação de *seed*, modelo e *prompt*.

In [ ]:
# Primeira amostragem: a seed controla o estado do gerador aleatório.
torch.manual_seed(42)

with torch.inference_mode():
    sampled_generated_ids = inference_model.generate(
        **input_data,
        do_sample=True,
        num_beams=1,
        max_new_tokens=MAX_NEW_TOKENS,
    )

sampled_new_ids = sampled_generated_ids[:, input_token_count:]
sampled_text = tokenizer.decode(sampled_new_ids[0], skip_special_tokens=True)

# Segunda amostragem: mesmo modelo e configurações, outra seed
torch.manual_seed(43)

with torch.inference_mode():
    sampled_generated_ids_2 = inference_model.generate(
        **input_data,
        do_sample=True,
        num_beams=1,
        max_new_tokens=MAX_NEW_TOKENS,
    )

sampled_new_ids_2 = sampled_generated_ids_2[:, input_token_count:]
sampled_text_2 = tokenizer.decode(sampled_new_ids_2[0], skip_special_tokens=True)

print("\n=== EXEMPLO 2: do_sample=True ===")
print(f"\nsemente (*seed*) 42 - tokens novos: {sampled_new_ids.shape[-1]}")
print(f"IDs gerados: {sampled_new_ids[0].tolist()}")
print(f"Texto decodificado:\n{sampled_text}")

print(f"\nsemente (*seed*) 43 - tokens novos: {sampled_new_ids_2.shape[-1]}")
print(f"IDs gerados: {sampled_new_ids_2[0].tolist()}")
print(f"Texto decodificado:\n{sampled_text_2}")

print("\n=== COMPARAÇÃO DAS ESTRATÉGIAS ===")
print(
    f"Amostragem (semente (*seed*) 42) igual à geração gulosa? {torch.equal(sampled_generated_ids, greedy_generated_ids)}"
)
print(
    f"Amostragens com semente (*seed*)s diferentes iguais? {torch.equal(sampled_generated_ids, sampled_generated_ids_2)}\n"
)



=== EXEMPLO 2: do_sample=True ===

semente (*seed*) 42 - tokens novos: 50
IDs gerados: [9660, 2022, 374, 279, 1882, 315, 14719, 1495, 1467, 1119, 3842, 4244, 476, 5766, 11, 892, 6147, 369, 803, 13382, 323, 22414, 8692, 315, 4128, 4119, 13, 1084, 17601, 24588, 323, 49445, 1817, 3409, 476, 3668, 2878, 264, 2661, 1467, 311, 27596, 5810, 4128, 8692, 9079, 1741, 438, 25975, 6358]
Texto decodificado:
 Tokenization is the process of breaking down text into individual words or characters, which allows for more accurate and meaningful processing of language models. It involves identifying and separating each word or character within a given text to facilitate natural language processing tasks such as sentiment analysis

semente (*seed*) 43 - tokens novos: 50
IDs gerados: [9660, 2022, 374, 279, 1882, 315, 14719, 1495, 1467, 1119, 3842, 4244, 476, 5766, 11, 892, 646, 387, 1483, 369, 5257, 9079, 1741, 438, 5810, 4128, 8692, 320, 45, 12567, 701, 5662, 6832, 11, 323, 1467, 23850, 13, 1084, 17601, 2

### Observações

Em `do_sample=False`, a decisão é uma escolha pelo maior *logit* ajustado em cada passo. Em `do_sample=True`, a decisão envolve um sorteio segundo uma distribuição de probabilidades, após os ajustes e filtros definidos na configuração do modelo. Em ambos os casos, o modelo continua produzindo a sequência de forma autorregressiva e `generate()` continua retornando IDs, não texto.

As duas estratégias compartilham `max_new_tokens=50`, que indica **limite máximo**, não quantidade obrigatória. Por isso, uma saída com menos de 50 tokens não contradiz o experimento: um token de fim de sequência pode encerrar a geração antes. O conteúdo textual deve ser examinado separadamente do controle de geração e não constitui, por si só, uma medida de qualidade, velocidade ou estabilidade.

**Decisão:** manter `do_sample=False` ao comparar configurações de CPU e GPU tem coerência com o objetivo de diminuir a variabilidade introduzida pela amostragem na sequência gerada. No entanto, isso **não elimina** outras fontes de variação experimental, como diferenças de precisão, ambiente ou execução. Se o objetivo futuro for estudar diversidade e comportamento de respostas, `do_sample=True` poderá ser avaliado em um experimento próprio.

## 3. Warmup e Cronometragem

TO-DO

## 4. Localização do modelo e carregamento

TO-DO

## 5. Memórias

TO-DO

## 6. Throughput e speedup

TO-DO

## 7. Preservação dos resultados

TO-DO

## 8. Recomendação para o Cliente

TO-DO

## 9. Fontes consultadas

https://huggingface.co/docs/transformers/main/en/glossary

https://qwen.ai/blog?id=qwen2.5-llm

https://huggingface.co/docs/transformers/v5.20.2/en/main_classes/tokenizer

https://savrn.com/models/qwen2-5-0-5b-instruct

https://savrn.com/models/compare/qwen2-0-5b-vs-qwen2-5-0-5b-instruct

https://llm-explorer.com/list/?ar=Qwen2ForCausalLM

https://deepgenerativemodels.github.io/notes/autoregressive/

https://developer.nvidia.com/blog/llm-benchmarking-fundamental-concepts/

https://machinelearningmastery.com/how-llms-choose-their-words-a-practical-walk-through-of-logits-softmax-and-sampling/

https://machinelearningmastery.com/a-gentle-introduction-to-attention-masking-in-transformer-models/